<a href="https://colab.research.google.com/github/praju-007anchan/IN126060102_NLP/blob/main/BERT_IMDB_Finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!ls

'BERT_IMDB_Finetune (1).ipynb'	 clean_notebook.ipynb   sample_data
 BERT_IMDB_Finetune.ipynb	 FINAL_CLEAN.ipynb


In [2]:
from google.colab import files
files.upload()

Saving BERT_IMDB_Finetune.ipynb to BERT_IMDB_Finetune (2).ipynb


{'BERT_IMDB_Finetune (2).ipynb': b'{\n  "nbformat": 4,\n  "nbformat_minor": 5,\n  "metadata": {\n    "colab": {\n      "provenance": [],\n      "gpuType": "T4"\n    },\n    "language_info": {\n      "name": "python"\n    },\n    "kernelspec": {\n      "name": "python3",\n      "display_name": "Python 3"\n    },\n    "accelerator": "GPU",\n    "widgets": {\n      "application/vnd.jupyter.widget-state+json": {\n        "c4104c47d4d94b20a1e06cd2cfa1bb61": {\n          "model_module": "@jupyter-widgets/controls",\n          "model_name": "HBoxModel",\n          "model_module_version": "1.5.0",\n          "state": {\n            "_dom_classes": [],\n            "_model_module": "@jupyter-widgets/controls",\n            "_model_module_version": "1.5.0",\n            "_model_name": "HBoxModel",\n            "_view_count": null,\n            "_view_module": "@jupyter-widgets/controls",\n            "_view_module_version": "1.5.0",\n            "_view_name": "HBoxView",\n            "box_style"

In [3]:
file_name = "BERT_IMDB_Finetune.ipynb"

In [4]:
import json

file_name = "BERT_IMDB_Finetune.ipynb"

with open(file_name, "r", encoding="utf-8") as f:
    data = json.load(f)

# 🔥 Remove notebook-level widgets
data.get("metadata", {}).pop("widgets", None)

# 🔥 Deep clean all cells
for cell in data.get("cells", []):
    metadata = cell.get("metadata", {})
    metadata.pop("widgets", None)

    # Remove widget outputs completely
    if "outputs" in cell:
        new_outputs = []
        for output in cell["outputs"]:
            if "data" in output:
                output["data"].pop("application/vnd.jupyter.widget-view+json", None)
            new_outputs.append(output)
        cell["outputs"] = new_outputs

# 🔥 FINAL SAFETY: remove any leftover keys globally
import re
clean_text = json.dumps(data)
clean_text = re.sub(r'"widgets"\s*:\s*{.*?}', '', clean_text)

data = json.loads(clean_text)

# Save final clean notebook
with open("FINAL_CLEAN.ipynb", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=1)

print("✅ FINAL CLEAN NOTEBOOK CREATED")

✅ FINAL CLEAN NOTEBOOK CREATED


In [5]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
!pip install transformers datasets -q

In [6]:
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoModel, BertTokenizerFast
from torch.optim import AdamW
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [7]:
dataset = load_dataset("imdb")
train_data = dataset["train"].shuffle(seed=42).select(range(5000))
test_data = dataset["test"].shuffle(seed=42).select(range(2000))
print(train_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvoyance". People like to compare, to judge, to evaluate. How about just enjoying? Funny thing too, people writing Fortier looks American but, on the other hand, arguing they prefer American series (!!!). Maybe it\'s the language, or the spirit, but I think this series is more English than American. By the way, the actors are really good and funny. The acting is not superficial at all...', 'label': 1}


In [8]:
from sklearn.model_selection import train_test_split
texts = [x['text'] for x in train_data]
labels = [x['label'] for x in train_data]
train_text, val_text, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.1, stratify=labels, random_state=42
)
test_text = [x['text'] for x in test_data]
test_labels = [x['label'] for x in test_data]

In [9]:
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
max_len = 128

tokens_train = tokenizer(train_text, max_length=max_len, padding=True, truncation=True)
tokens_val = tokenizer(val_text, max_length=max_len, padding=True, truncation=True)
tokens_test = tokenizer(test_text, max_length=max_len, padding=True, truncation=True)

train_seq = torch.tensor(tokens_train['input_ids'])
train_mask = torch.tensor(tokens_train['attention_mask'])
train_y = torch.tensor(train_labels)

val_seq = torch.tensor(tokens_val['input_ids'])
val_mask = torch.tensor(tokens_val['attention_mask'])
val_y = torch.tensor(val_labels)

test_seq = torch.tensor(tokens_test['input_ids'])
test_mask = torch.tensor(tokens_test['attention_mask'])
test_y = torch.tensor(test_labels)

In [10]:
batch_size = 16
train_loader = DataLoader(TensorDataset(train_seq, train_mask, train_y), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(val_seq, val_mask, val_y), batch_size=batch_size)

In [11]:
bert = AutoModel.from_pretrained('bert-base-uncased')
for param in bert.parameters():
    param.requires_grad = False

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
class BERT_Arch(nn.Module):
    def __init__(self, bert):
        super().__init__()
        self.bert = bert
        self.fc1 = nn.Linear(768,256)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(256,2)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, sent_id, mask):
        _, cls = self.bert(sent_id, attention_mask=mask, return_dict=False)
        x = self.fc1(cls)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.logsoftmax(x)

model = BERT_Arch(bert).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
class_wts = compute_class_weight(class_weight='balanced', classes=np.unique(train_labels), y=train_labels)
weights = torch.tensor(class_wts, dtype=torch.float).to(device)
loss_fn = nn.NLLLoss(weight=weights)

In [13]:
def train():
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = [b.to(device) for b in batch]
        sent_id, mask, labels = batch
        model.zero_grad()
        preds = model(sent_id, mask)
        loss = loss_fn(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

def evaluate():
    model.eval()
    total_loss = 0
    preds_all = []
    with torch.no_grad():
        for batch in val_loader:
            batch = [b.to(device) for b in batch]
            sent_id, mask, labels = batch
            preds = model(sent_id, mask)
            loss = loss_fn(preds, labels)
            total_loss += loss.item()
            preds_all.append(preds.cpu().numpy())
    return total_loss / len(val_loader), np.concatenate(preds_all)

In [14]:
epochs = 2
for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}")

    train_loss = train()
    val_loss, _ = evaluate()

    print("Train Loss:", round(train_loss, 3))
    print("Val Loss:", round(val_loss, 3))


Epoch 1
Train Loss: 0.686
Val Loss: 0.68

Epoch 2
Train Loss: 0.674
Val Loss: 0.678


In [15]:
model.eval()
with torch.no_grad():
    preds = model(test_seq.to(device), test_mask.to(device))
    preds = preds.cpu().numpy()
preds = np.argmax(preds, axis=1)
print(classification_report(test_y, preds))
cm = confusion_matrix(test_y, preds)
print(cm)

              precision    recall  f1-score   support

           0       0.73      0.32      0.44      1000
           1       0.56      0.88      0.69      1000

    accuracy                           0.60      2000
   macro avg       0.65      0.60      0.57      2000
weighted avg       0.65      0.60      0.57      2000

[[318 682]
 [118 882]]
